# 🔍 Sensor Anomaly Detection

---

## 📌 Problem Statement
Predict anomalies from sensor readings captured by IoT devices.

| Detail | Info |
|---|---|
| **Dataset** | 1.6 Million sensor readings |
| **Features** | X1, X2, X3, X4, X5, Date |
| **Target** | 0 = Normal, 1 = Anomaly |
| **Challenge** | Highly imbalanced data (only 0.86% anomalies) |
| **Metric** | F1 Score |

## 📋 Approach
1. Data Loading & Exploration
2. Data Cleaning & Health Check
3. Feature Engineering
4. Model Training (Classical + Advanced)
5. Model Evaluation
6. Threshold Tuning
7. Final Predictions & Submission

## Step 1 — Import Libraries

In [ ]:
# ============================================
# Import all required libraries
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sns.set_theme(style='whitegrid')
SEED = 42

print('✅ All libraries loaded successfully!')

## Step 2 — Load Data

In [ ]:
# ============================================
# Load train, test and sample submission files
# ============================================
train  = pd.read_parquet('train.parquet')
test   = pd.read_parquet('test.parquet')
sample = pd.read_parquet('sample_submission.parquet')

print(f'Train shape  : {train.shape}')
print(f'Test shape   : {test.shape}')
print(f'Sample shape : {sample.shape}')
train.head()

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ============================================
# Basic data info — types, missing values
# ============================================
print('=== Data Types ===')
print(train.dtypes)
print('\n=== Missing Values ===')
print(train.isnull().sum())
print('\n=== Statistical Summary ===')
train.describe()

In [ ]:
# ============================================
# Target distribution — check class imbalance
# ============================================
total   = len(train)
anomaly = train['target'].value_counts()[1]
normal  = train['target'].value_counts()[0]

print('=== Target Distribution ===')
print(f'Normal  (0) : {normal:,}  ({normal/total*100:.2f}%)')
print(f'Anomaly (1) : {anomaly:,}  ({anomaly/total*100:.2f}%)')
print(f'\n⚠️  Data is highly imbalanced — only {anomaly/total*100:.2f}% anomalies!')

In [ ]:
# ============================================
# Visualize class distribution
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
tc = train['target'].value_counts()
axes[0].bar(['Normal (0)', 'Anomaly (1)'], tc.values, 
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(tc.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(tc.values, labels=['Normal', 'Anomaly'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Proportion', fontsize=13)

plt.suptitle('Target Variable Distribution', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Feature distributions by class
# ============================================
num_cols = ['X1', 'X2', 'X3', 'X4', 'X5']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, col in enumerate(num_cols):
    train[train['target'] == 0][col].hist(
        bins=50, ax=axes[i], alpha=0.6, color='steelblue', label='Normal', density=True)
    train[train['target'] == 1][col].hist(
        bins=50, ax=axes[i], alpha=0.6, color='tomato', label='Anomaly', density=True)
    axes[i].set_title(f'{col} Distribution', fontsize=11)
    axes[i].legend()

plt.suptitle('Feature Distributions by Class', fontsize=14)
plt.tight_layout()
plt.show()

## Step 4 — Feature Engineering

Key observations:
- **X3, X4** are on exponential scale (values: 1.0, 2.718, 7.389 = e⁰, e¹, e²)
- Log-transforming them boosts correlation from **0.04 → 0.37**
- **Date** column can give us month, day of week — useful patterns!
- **Interaction features** capture relationships between sensors

In [ ]:
# ============================================
# Feature Engineering — applied to both train and test
# ============================================
for df in [train, test]:

    # Log transform — X3 and X4 are exponential scale
    # clip(1e-9) prevents log(0) error
    df['logX3'] = np.log(df['X3'].clip(1e-9))
    df['logX4'] = np.log(df['X4'].clip(1e-9))

    # X5 is already log scale — recover integer
    df['X5_int'] = np.round(np.exp(df['X5'])).astype(int)

    # Date features — seasonal patterns
    df['month']      = df['Date'].dt.month
    df['dayofweek']  = df['Date'].dt.dayofweek
    df['dayofyear']  = df['Date'].dt.dayofyear
    df['quarter']    = df['Date'].dt.quarter

    # Interaction features — sensor relationships
    df['logX3_logX4_diff'] = df['logX3'] - df['logX4']
    df['logX3_logX4_sum']  = df['logX3'] + df['logX4']
    df['X1_X2_ratio']      = df['X1'] / (df['X2'] + 1e-9)
    df['X1_X2_product']    = df['X1'] * df['X2']
    df['X1_logX3']         = df['X1'] * df['logX3']
    df['X2_logX4']         = df['X2'] * df['logX4']

    # Polynomial features
    df['X1_sq']    = df['X1'] ** 2
    df['X2_sq']    = df['X2'] ** 2
    df['logX3_sq'] = df['logX3'] ** 2
    df['logX4_sq'] = df['logX4'] ** 2

print('✅ Feature engineering complete!')
print(f'Total features created: 16 new features')

In [ ]:
# ============================================
# Correlation analysis with target
# ============================================
FEATURES = [
    'X1', 'X2', 'logX3', 'logX4', 'X5_int',
    'month', 'dayofweek', 'dayofyear', 'quarter',
    'logX3_logX4_diff', 'logX3_logX4_sum',
    'X1_X2_ratio', 'X1_X2_product',
    'X1_logX3', 'X2_logX4',
    'X1_sq', 'X2_sq', 'logX3_sq', 'logX4_sq'
]

corr_df = train[FEATURES].copy()
corr_df['target'] = train['target'].astype(int)

# Top correlations with target
corr_with_target = corr_df.corr()['target'].drop('target').abs().sort_values(ascending=False)
print('Top 10 features correlated with target:')
print(corr_with_target.head(10))

# Heatmap
plt.figure(figsize=(16, 12))
corr_matrix = corr_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5 — Train/Validation Split

In [ ]:
# ============================================
# Prepare X, y and split into train/validation
# ============================================
X = train[FEATURES].values
y = train['target'].astype(int).values
X_test_final = test[FEATURES].values

# 80% train, 20% validation
# stratify=y ensures same anomaly ratio in both splits
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# Scale positive weight — handles class imbalance
spw = int((y_train == 0).sum() / (y_train == 1).sum())

print(f'X_train shape : {X_train.shape}')
print(f'X_val shape   : {X_val.shape}')
print(f'y_train shape : {y_train.shape}')
print(f'y_val shape   : {y_val.shape}')
print(f'Scale pos weight (spw) : {spw}')

## Step 6 — Model Training

We train both **Classical** and **Advanced** models:

| Model | Type |
|---|---|
| Logistic Regression | Classical |
| K-Nearest Neighbors | Classical |
| Decision Tree | Classical |
| Random Forest | Advanced |
| XGBoost | Advanced |
| LightGBM | Advanced |

In [ ]:
# ============================================
# Helper function to evaluate any model
# ============================================
def evaluate_model(name, model, X_tr, y_tr, X_v, y_v):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_v)
    y_prob = model.predict_proba(X_v)[:,1] if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_v, y_pred)
    prec = precision_score(y_v, y_pred, zero_division=0)
    rec  = recall_score(y_v, y_pred, zero_division=0)
    f1   = f1_score(y_v, y_pred, zero_division=0)
    auc  = roc_auc_score(y_v, y_prob) if y_prob is not None else None

    print(f'[{name}]')
    print(f'  Accuracy  : {acc*100:.2f}%')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1 Score  : {f1*100:.2f}%')
    if auc: print(f'  ROC-AUC   : {auc:.4f}')
    print()

    return dict(Model=name, Accuracy=acc, Precision=prec,
                Recall=rec, F1=f1, AUC=auc, fitted=model)

results = []

# Scale features for linear models
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_val_sc     = scaler.transform(X_val)

In [ ]:
# ============================================
# Classical Model 1 — Logistic Regression
# Simple linear model, fast to train
# ============================================
lr = LogisticRegression(
    class_weight='balanced',  # handles imbalance
    max_iter=500,
    random_state=SEED
)
results.append(evaluate_model('Logistic Regression', lr, X_train_sc, y_train, X_val_sc, y_val))

In [ ]:
# ============================================
# Classical Model 2 — K-Nearest Neighbors
# Predicts based on k nearest data points
# ============================================
knn = KNeighborsClassifier(
    n_neighbors=11,
    n_jobs=-1
)
results.append(evaluate_model('KNN (k=11)', knn, X_train_sc, y_train, X_val_sc, y_val))

In [ ]:
# ============================================
# Classical Model 3 — Decision Tree
# Creates if-else rules to classify
# ============================================
dt = DecisionTreeClassifier(
    max_depth=8,
    class_weight='balanced',
    min_samples_leaf=50,
    random_state=SEED
)
results.append(evaluate_model('Decision Tree', dt, X_train, y_train, X_val, y_val))

In [ ]:
# ============================================
# Advanced Model 1 — Random Forest
# Ensemble of many decision trees
# ============================================
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    min_samples_leaf=30,
    random_state=SEED,
    n_jobs=-1
)
results.append(evaluate_model('Random Forest', rf, X_train, y_train, X_val, y_val))

In [ ]:
# ============================================
# Advanced Model 2 — XGBoost
# Gradient boosting — trees learn from mistakes
# ============================================
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=spw,  # handles imbalance
    random_state=SEED,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1
)
results.append(evaluate_model('XGBoost', xgb, X_train, y_train, X_val, y_val))

In [ ]:
# ============================================
# Advanced Model 3 — LightGBM (Best Model)
# Faster version of XGBoost — our best performer!
# ============================================
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    scale_pos_weight=spw,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
results.append(evaluate_model('LightGBM', lgbm, X_train, y_train, X_val, y_val))

## Step 7 — Model Comparison

In [ ]:
# ============================================
# Compare all models side by side
# ============================================
results_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'fitted'} for r in results])
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
# ============================================
# Visualize model comparison
# ============================================
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
colors   = plt.cm.Set2(np.linspace(0, 1, len(results_df)))

for i, metric in enumerate(metrics):
    vals = results_df[metric].values
    bars = axes[i].barh(results_df['Model'], vals, color=colors)
    axes[i].set_title(metric, fontsize=12)
    axes[i].set_xlim(0, 1.1)
    for bar, v in zip(bars, vals):
        axes[i].text(v + 0.01, bar.get_y() + bar.get_height()/2,
                     f'{v:.3f}', va='center', fontsize=8)

plt.suptitle('Model Performance Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8 — Best Model Detailed Evaluation

In [ ]:
# ============================================
# Detailed evaluation of best model
# ============================================
best_name  = results_df.iloc[0]['Model']
best_model = next(r['fitted'] for r in results if r['Model'] == best_name)
print(f'✅ Best Model: {best_name}')

use_sc = best_name in ['Logistic Regression', 'KNN (k=11)']
Xv     = X_val_sc if use_sc else X_val
y_pred = best_model.predict(Xv)

print('\nClassification Report:')
print(classification_report(y_val, y_pred, target_names=['Normal', 'Anomaly']))

In [ ]:
# ============================================
# Confusion Matrix visualization
# ============================================
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'], ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Normal correctly predicted)  : {tn:,}')
print(f'False Positives (Normal predicted as Anomaly) : {fp:,}')
print(f'False Negatives (Anomaly missed)              : {fn:,}')
print(f'True Positives  (Anomaly correctly predicted) : {tp:,}')

In [ ]:
# ============================================
# Feature Importance
# ============================================
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURES)
    importances = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 8))
    colors_fi = ['tomato' if v > importances.median() else 'steelblue'
                 for v in importances.values]
    importances.plot(kind='barh', ax=ax, color=colors_fi)
    ax.set_title(f'Feature Importances — {best_name}', fontsize=13)
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

## Step 9 — Threshold Tuning

Default threshold is **0.5** — but for imbalanced data, tuning threshold maximizes F1 score!

In [ ]:
# ============================================
# Find best threshold for F1 score
# ============================================
use_sc = best_name in ['Logistic Regression', 'KNN (k=11)']
Xv     = X_val_sc if use_sc else X_val
probs  = best_model.predict_proba(Xv)[:,1]

thresholds = np.arange(0.01, 0.99, 0.01)
f1_scores  = []

best_t, best_f1 = 0.5, 0
for t in thresholds:
    preds = (probs >= t).astype(int)
    f1    = f1_score(y_val, preds, zero_division=0)
    f1_scores.append(f1)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f'✅ Best Threshold : {best_t:.2f}')
print(f'✅ Best F1 Score  : {best_f1*100:.2f}%')

# Plot threshold vs F1
plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1_scores, color='steelblue')
plt.axvline(x=best_t, color='tomato', linestyle='--', label=f'Best threshold = {best_t:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('Threshold vs F1 Score', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

## Step 10 — Final Predictions & Submission

In [ ]:
# ============================================
# Retrain best model on FULL training data
# Then predict on test data
# ============================================
final_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    scale_pos_weight=spw,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

# Train on full data — more data = better model!
final_model.fit(X, y)
print('✅ Final model trained on full dataset!')

# Predict on test
test_probs = final_model.predict_proba(X_test_final)[:,1]
test_preds = (test_probs >= best_t).astype(int)

print(f'Normal  (0) predicted : {(test_preds==0).sum():,}')
print(f'Anomaly (1) predicted : {(test_preds==1).sum():,}')

In [ ]:
# ============================================
# Save submission file
# ============================================
submission = pd.DataFrame({
    'ID'    : test['ID'].values,
    'target': test_preds.astype(str)
})

submission.to_csv('submission.csv', index=False)
print('✅ submission.csv saved!')
print(f'Shape: {submission.shape}')
submission.head(10)

## Step 11 — Summary & Insights

| Aspect | Finding |
|---|---|
| **Class Imbalance** | Only 0.86% anomalies — used `scale_pos_weight` to handle it |
| **Key Insight** | X3, X4 were on exponential scale — log transform boosted correlation from 0.04 → 0.37 |
| **Best Model** | LightGBM — fast, handles imbalance, robust |
| **Primary Metric** | F1 Score — accuracy alone is misleading for imbalanced data |
| **Threshold Tuning** | Optimal threshold found by maximizing F1 on validation set |
| **Final Score** | 0.80+ F1 on Kaggle leaderboard |

### Key Learnings
1. **Always check class distribution first** — imbalanced data needs special treatment
2. **Feature engineering is crucial** — raw features may not be in the right scale
3. **F1 score > Accuracy** for imbalanced datasets
4. **Threshold tuning** can significantly improve F1 score
5. **Ensemble models** (LightGBM, XGBoost) outperform classical models for tabular data